### Step 1: Configure imports and helpers

Load the required Python modules, repository paths, database constants, table list, and helper functions used by the rest of the proof notebook.

In [1]:
import os
import subprocess
from pathlib import Path

from flask_sqlalchemy import SQLAlchemy
from sqlalchemy import create_engine, event

from quantdb.api import make_app
from quantdb.extract_microct import MICROCT_UUID
from quantdb.ingest_microct import (
    _count_microct,
    delete_microct_data,
    ingest_microct,
)
from quantdb.models import reflect_models
from quantdb.utils import dbUri

REPO = Path("/Users/tmsincomb/Dropbox (Personal)/repos/quantitative-db")
SQL = REPO / "sql"
DUMP = REPO / "resources" / "quantdb_demo.dump"
DB = "quantdb_demo"
PG_DUMP = "/opt/homebrew/opt/postgresql@16/bin/pg_dump"
PG_RESTORE = "/opt/homebrew/opt/postgresql@16/bin/pg_restore"
AWS_HOST = "troy-quantdb-test.crxhhfokqjgu.us-east-1.rds.amazonaws.com"
AWS_DB = "postgres"
AWS_USER = "postgres"
DATASET = MICROCT_UUID
OBJ = "aaaaaaaa-1111-2222-3333-444444444401"
DATA_TABLES = [
    "objects",
    "dataset_object",
    "values_inst",
    "instance_parent",
    "obj_desc_inst",
    "obj_desc_quant",
    "obj_desc_cat",
    "values_quant",
    "values_cat",
]


def psql(sql=None, *, file=None, db="postgres", extra=None):
    cmd = ["psql", "-U", "postgres", "-h", "localhost", "-p", "5432", "-d", db, "-v", "ON_ERROR_STOP=on"]
    if extra:
        for k, v in extra.items():
            cmd += ["-v", f"{k}={v}"]
    cmd += ["-f", str(file)] if file else ["-c", sql]
    subprocess.run(cmd, check=True, capture_output=True, text=True)


def set_search_path(dbapi_conn, _record):
    cur = dbapi_conn.cursor()
    cur.execute("SET search_path TO quantdb, public")
    cur.close()

### Step 2: Rebuild the local demo database

Create a clean `quantdb_demo` PostgreSQL database, load the base schema and lookup data, then reflect the SQLAlchemy models.

In [2]:
psql(file=SQL / "postgres.sql", extra={"test_database": DB, "database": DB})
psql(sql='GRANT "quantdb-test-admin" TO CURRENT_USER;')
psql(sql=f"DROP DATABASE IF EXISTS {DB};")
psql(
    sql=(
        f"CREATE DATABASE {DB} WITH OWNER='quantdb-test-admin' "
        f"TEMPLATE template0 ENCODING='UTF8' "
        f"LC_COLLATE='C' LC_CTYPE='C';"
    )
)
psql(sql='REVOKE "quantdb-test-admin" FROM CURRENT_USER;')
psql(sql="ALTER ROLE postgres SET search_path = quantdb, public;")
psql(file=SQL / "schemas.sql", db=DB)
psql(file=SQL / "tables.sql", db=DB)
psql(file=SQL / "permissions.sql", db=DB, extra={"database": DB, "perm_user": "quantdb-test-user"})
psql(file=SQL / "inserts.sql", db=DB)
psql(file=SQL / "inserts_microct.sql", db=DB)

engine = create_engine(dbUri("postgres", "localhost", 5432, DB))
event.listen(engine, "connect", set_search_path)
models = reflect_models(engine=engine)
len(models.Base.classes)

15

### Step 3: Ingest a minimal MicroCT payload

Build a small representative MicroCT object graph and load it into the local demo database through `ingest_microct()`.

In [3]:
PAYLOAD = {
    "session": models.Session(),
    "models": models,
    "data_dicts": {
        "objects": [
            {"id": DATASET, "id_type": "dataset", "id_file": None},
            {"id": OBJ, "id_type": "package", "id_file": 1},
        ],
        "dataset_object": [{"dataset": DATASET, "object": OBJ}],
        "values_inst": [
            {"dataset": DATASET, "id_formal": "sub-X", "type": "subject", "desc_inst": "human", "id_sub": "sub-X"},
            {
                "dataset": DATASET,
                "id_formal": "sam-X-1",
                "type": "sample",
                "desc_inst": "tissue",
                "id_sub": "sub-X",
                "id_sam": "sam-X-1",
            },
            {
                "dataset": DATASET,
                "id_formal": "nerve-X-1",
                "type": "below",
                "desc_inst": "nerve",
                "id_sub": "sub-X",
                "id_sam": "sam-X-1",
            },
            {
                "dataset": DATASET,
                "id_formal": "nerve-X-1-slice-0",
                "type": "below",
                "desc_inst": "nerve-cross-section",
                "id_sub": "sub-X",
                "id_sam": "sam-X-1",
            },
        ],
        "instance_parent": [
            {"id": {"dataset": DATASET, "id_formal": "sam-X-1"}, "parent": {"dataset": DATASET, "id_formal": "sub-X"}},
            {
                "id": {"dataset": DATASET, "id_formal": "nerve-X-1"},
                "parent": {"dataset": DATASET, "id_formal": "sam-X-1"},
            },
            {
                "id": {"dataset": DATASET, "id_formal": "nerve-X-1-slice-0"},
                "parent": {"dataset": DATASET, "id_formal": "nerve-X-1"},
            },
        ],
        "values_quant": [
            {
                "value": 1.0,
                "value_blob": 1.0,
                "object": OBJ,
                "desc_inst": "nerve-cross-section",
                "desc_quant": "nerve cross section diameter um",
                "instance": {"dataset": DATASET, "id_formal": "nerve-X-1-slice-0"},
            },
        ],
        "values_cat": [],
    },
}
ingest_microct(**PAYLOAD)
PAYLOAD["session"].commit()
PAYLOAD["session"].close()
{k: len(v) for k, v in PAYLOAD["data_dicts"].items()}

/Users/tmsincomb/Dropbox (Personal)/repos/quantitative-db/quantdb/ingest_microct.py:348: SAWarning: relationship 'DescriptorsInst.valuesquant_via_desc_inst_collection' will copy column descriptors_inst.id to column values_quant.desc_inst, which conflicts with relationship(s): 'ObjDescInst.valuesquant_collection' (copies obj_desc_inst.desc_inst to values_quant.desc_inst), 'ValuesQuant.objdescinst' (copies obj_desc_inst.desc_inst to values_quant.desc_inst). If this is not the intention, consider if these relationships should be linked with back_populates, or if viewonly=True should be applied to one or more if they are read-only. For the less common case that foreign key constraints are partially overlapping, the orm.foreign() annotation can be used to isolate the columns that should be written towards.   To silence this warning, add the parameter 'overlaps="objdescinst,valuesquant_collection"' to the 'DescriptorsInst.valuesquant_via_desc_inst_collection' relationship. (Background on thi

{'objects': 2,
 'dataset_object': 1,
 'values_inst': 4,
 'instance_parent': 3,
 'values_quant': 1,
 'values_cat': 0}

### Step 4: Confirm local ingest counts

Query the local demo database and assert that the expected MicroCT objects, instances, hierarchy rows, and measurement rows were created.

In [4]:
s = models.Session()
counts = _count_microct(s, models)
assert counts["objects"] >= 2
assert counts["dataset_object"] >= 1
assert counts["values_inst"] >= 4
assert counts["instance_parent"] >= 3
assert counts["values_quant"] >= 1
s.close()
counts

{'values_inst': 4,
 'dataset_object': 1,
 'instance_parent': 3,
 'objects': 2,
 'values_quant': 1,
 'values_cat': 0,
 'obj_desc_inst': 1,
 'obj_desc_quant': 1,
 'obj_desc_cat': 0}

### Step 5: Publish the demo slice to AWS

Dump the local demo tables, prepare the AWS schema, remove any previous MicroCT demo rows, and restore the fresh demo data.

In [7]:
DUMP.parent.mkdir(parents=True, exist_ok=True)
dump_cmd = [PG_DUMP, "--data-only", "-Fc", "-U", "postgres", "-h", "localhost", "-p", "5432", "-d", DB, "-f", str(DUMP)]
for t in DATA_TABLES:
    dump_cmd += ["-t", f"quantdb.{t}"]
subprocess.run(dump_cmd, check=True)

env = {**os.environ, "PGSSLMODE": "require"}
subprocess.run(["bash", str(REPO / "bin" / "aws_setup")], check=True, env=env)

aws_engine = create_engine(
    dbUri(AWS_USER, AWS_HOST, 5432, AWS_DB),
    connect_args={"sslmode": "require"},
)
event.listen(aws_engine, "connect", set_search_path)
aws_models = reflect_models(engine=aws_engine)

sa = aws_models.Session()
delete_microct_data(sa, aws_models)
sa.commit()
sa.close()

restore_proc = subprocess.Popen([PG_RESTORE, "-f", "-", str(DUMP)], stdout=subprocess.PIPE, env=env)
filtered = subprocess.Popen(
    ["grep", "-vE", r"^(SELECT pg_catalog\.set_config\('search_path', ''|\\restrict|\\unrestrict)"],
    stdin=restore_proc.stdout,
    stdout=subprocess.PIPE,
)
restore_proc.stdout.close()
psql_env = {**env, "PGOPTIONS": "-c search_path=quantdb,public"}
psql_proc = subprocess.run(
    ["psql", "-U", AWS_USER, "-h", AWS_HOST, "-p", "5432", "-d", AWS_DB, "-v", "ON_ERROR_STOP=on"],
    stdin=filtered.stdout,
    env=psql_env,
    check=True,
)
filtered.stdout.close()
DUMP.stat().st_size

=== AWS Setup: quantdb schema on troy-quantdb-test.crxhhfokqjgu.us-east-1.rds.amazonaws.com ===
Step 1: Creating roles...
DO
ALTER ROLE
ALTER ROLE
ALTER ROLE
ALTER ROLE
Step 2: Setting search_path...
ALTER ROLE
Step 3: Creating schema...


psql:/Users/tmsincomb/Dropbox (Personal)/repos/quantitative-db/sql/schemas.sql:3: NOTICE:  schema "quantdb" already exists, skipping


CREATE SCHEMA
Step 4: Tables already exist (20 tables), skipping.
Step 5: Granting permissions...
GRANT
GRANT
GRANT
GRANT
Step 6: Lookup data already loaded (8 units), skipping.
=== AWS Setup complete ===


/Users/tmsincomb/Dropbox (Personal)/repos/quantitative-db/quantdb/ingest_microct.py:245: SAWarning: relationship 'ObjDescInst.valuesquant_collection' will copy column obj_desc_inst.desc_inst to column values_quant.desc_inst, which conflicts with relationship(s): 'DescriptorsInst.valuesquant_via_desc_inst_collection' (copies descriptors_inst.id to values_quant.desc_inst), 'ValuesQuant.descriptorsinst_via_desc_inst' (copies descriptors_inst.id to values_quant.desc_inst). If this is not the intention, consider if these relationships should be linked with back_populates, or if viewonly=True should be applied to one or more if they are read-only. For the less common case that foreign key constraints are partially overlapping, the orm.foreign() annotation can be used to isolate the columns that should be written towards.   To silence this warning, add the parameter 'overlaps="descriptorsinst_via_desc_inst,valuesquant_via_desc_inst_collection"' to the 'ObjDescInst.valuesquant_collection' rela

ProgrammingError: (psycopg2.errors.UndefinedFunction) operator does not exist: uuid = text
LINE 1: DELETE FROM quantdb.objects WHERE id = ANY(ARRAY['fb1cbd05-4...
                                             ^
HINT:  No operator matches the given name and argument types. You might need to add explicit type casts.

[SQL: DELETE FROM quantdb.objects WHERE id = ANY(%(uuids)s)]
[parameters: {'uuids': ['fb1cbd05-4320-4d8b-ac3a-44f1fe810718']}]
(Background on this error at: https://sqlalche.me/e/20/f405)

### Step 6: Verify the AWS-backed API

Point the Flask test client at the AWS test database and confirm the public API endpoints respond successfully.

In [ ]:
os.environ["QUANTDB_DB_HOST"] = AWS_HOST
os.environ["QUANTDB_DB_PORT"] = "5432"
os.environ["QUANTDB_DB_USER"] = AWS_USER
os.environ["QUANTDB_DB_DATABASE"] = AWS_DB
os.environ["PGSSLMODE"] = "require"

app = make_app(db=SQLAlchemy(), test=False)
client = app.test_client()
r1 = client.get("/api/1//db-name")
assert r1.status_code == 200
assert AWS_DB in r1.get_data(as_text=True)
r2 = client.get("/api/1//classes?include-unused=true")
assert r2.status_code == 200
(r1.status_code, r1.get_data(as_text=True), r2.status_code, len(r2.get_data()))